# Chapter 3 Exercise Solutions

Worked answers to the 3 exercises in Section 3.11. Each answer ends with the judgment call that matters on real data.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "ch03_data":
    DATA = ROOT / "output_data" / "generated_data"
else:
    DATA = ROOT / "ch03_data" / "output_data" / "generated_data"
print(DATA.resolve())

/Users/qiu/Projects/hands-on-pharma-decision-science/ch03_data/output_data/generated_data


## Exercise 1: Beat the answer key

Drop `paid_amount` from the duplicate key and measure the over-match.

In [2]:
mc = pd.read_csv(DATA / "claims_medical" / "medical_claims.csv")
full_key = ["patient_id", "claim_date", "icd10_code", "procedure_code", "hcp_id", "paid_amount"]
short_key = full_key[:-1]
full_count = int(mc.duplicated(full_key, keep="first").sum())
short_count = int(mc.duplicated(short_key, keep="first").sum())
print(f"full_key_duplicates={full_count:,}")
print(f"short_key_duplicates={short_count:,}")
print(f"false_duplicates_after_dropping_paid_amount={short_count-full_count:,}")

full_key_duplicates=1,002
short_key_duplicates=1,011
false_duplicates_after_dropping_paid_amount=9


**Methods note:** the shorter key flags 9 legitimate same-day records as duplicates. In production, inspect over-matched pairs and document why each field belongs in the key. A duplicate rule is a business definition, not a pandas default.

## Exercise 2: Did the access change move utilization?

Compare PAY001 Roventra rejection rates before and after the April 1 tier change.

In [3]:
rx = pd.read_csv(DATA / "claims_pharmacy" / "pharmacy_claims.csv", dtype={"ndc": str}, parse_dates=["fill_date"])
rows = rx.loc[rx.payer_id.eq("PAY001") & rx.ndc.eq("90000-1001-11") & rx.status.isin(["Rejected", "Paid"])].copy()
rows["period"] = pd.cut(rows.fill_date, pd.to_datetime(["2023-12-31", "2024-03-31", "2024-09-30"]), labels=["Q1", "Q2-Q3"])
result = rows.groupby("period", observed=True).status.agg(transactions="size", rejected=lambda s: s.eq("Rejected").sum())
result["rejection_rate_pct"] = (100 * result.rejected / result.transactions).round(1)
print(result.to_string())

        transactions  rejected  rejection_rate_pct
period                                            
Q1               187        53                28.3
Q2-Q3           1441       193                13.4


**Methods note:** the rejection rate falls from 28.3% in Q1 to 13.4% in Q2-Q3, but this comparison does not establish an effect. Patient mix, seasonality, launch learning, other payer rules, and changing transaction volume could explain the difference. A causal claim needs a prespecified comparison design.

## Exercise 3: Interrogate PSA missingness

Measure PSA testing by condition bucket and sex.

In [4]:
patients = pd.read_csv(DATA / "reference" / "patients.csv")
ehr = pd.read_csv(DATA / "ehr" / "ehr_events.csv")
psa_patients = set(ehr.loc[ehr.event_name.eq("PSA"), "patient_id"])
patients["psa_tested"] = patients.patient_id.isin(psa_patients)
result = (patients.groupby(["condition_bucket", "sex"]).psa_tested
    .agg(tested="sum", patients="size"))
result["testing_rate_pct"] = (100 * result.tested / result.patients).round(1)
print(result.to_string())

                      tested  patients  testing_rate_pct
condition_bucket sex                                    
Cardiology       F         0      1531               0.0
                 M         0      1611               0.0
Launch condition F         0      4655               0.0
                 M         0      4562               0.0
Oncology         F         0      1464               0.0
                 M      1441      1598              90.2
Other            F         0      1512               0.0
                 M         0      1494               0.0
Rheumatology     F         0       772               0.0
                 M         0       801               0.0


**Methods note:** testing occurs only among male patients in the oncology bucket, and about 90% of that group is tested. On real data, this pattern would rule out a missing-at-random assumption. The model should represent testing eligibility and the test-ordering process before using the result value.